# Сравнение Logistic Regression

Отдельные таблицы fixed C=1 и tuned C; одинаковые folds по людям. OOF использован для выбора; выданная val — последующая проверка, а при повторном выборе по ней становится частью разработки. SMOKE-метрики не являются оценкой полного эксперимента. Для этого notebook используйте ядро окружения проекта.

In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.data import STAGES
from src.evaluation import comparison_table

OUTPUT_ROOT = PROJECT_ROOT / "artefacts"
SMOKE = False
RESULT_ROOT = OUTPUT_ROOT / "smoke" if SMOKE else OUTPUT_ROOT
CONFIG_IDS = [
    path.stem for path in sorted((PROJECT_ROOT / "configs").glob("*_linear.yaml"))
]
IDS = [
    name
    for name in CONFIG_IDS
    if (RESULT_ROOT / name / "metrics.json").is_file()
    and "val" in json.loads((RESULT_ROOT / name / "metrics.json").read_text())
]
missing = [name for name in CONFIG_IDS if name not in IDS]
if missing:
    print("Нет OOF/val-оценки (незавершённые эксперименты):", ", ".join(missing))
if not IDS:
    raise FileNotFoundError(f"Нет завершённых экспериментов в {RESULT_ROOT}")
comparison = comparison_table(RESULT_ROOT, IDS)
for regime in ("fixed", "tuned"):
    print(regime, "SMOKE" if SMOKE else "full train/val")
    display(pd.DataFrame(comparison.filter(comparison["regime"] == regime).to_dicts()))

In [ ]:
EXPERIMENT_ID = "001_linear"
folder = RESULT_ROOT / EXPERIMENT_ID
metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
search = pd.read_csv(folder / "search.csv")
display(search)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row, regime in enumerate(("fixed", "tuned")):
    for col, scope in enumerate(("oof", "val")):
        values = metrics[regime]["oof"] if scope == "oof" else metrics["val"][regime]
        sns.heatmap(
            values["confusion_matrix"],
            annot=True,
            fmt="d",
            xticklabels=STAGES,
            yticklabels=STAGES,
            ax=axes[row, col],
            cmap="Blues",
        )
        axes[row, col].set(
            title=f"{EXPERIMENT_ID}: {regime} / {scope}",
            xlabel="Predicted",
            ylabel="True",
        )
plt.tight_layout()
plt.show()
for regime in ("fixed", "tuned"):
    print(regime, "per-subject val (five-class macro-F1, including absent classes)")
    display(pd.DataFrame(metrics["val"][regime]["subjects"]))

Вероятности с ID находятся в fixed_predictions.npz / tuned_predictions.npz; val_probabilities и oof разделены. Для ансамбля используем согласованные OOF-вероятности, а не обучающие предсказания. Пока ансамбль и коррекция вероятностей не выполняются.